# Cross-cultural Adaptation and Psychometric Validation of the Perceived Quality of University Experiences Scale (PQUES) among Business Students in Chile

**Supplementary material — Content validity analysis (Aiken's V)**

|              |                                                        |
|--------------|--------------------------------------------------------|
| **Authors**  | R. Monge; [Author 2]; [Author 3]; [Author 4]           |
| **Contact**  | <corresponding.author@institution.edu>                 |
| **Date**     | January 12, 2026                                       |
| **Environment** | Google Colab (Python 3)                             |

*(Replace the bracketed placeholders with the full author list.)*

**Purpose.** Quantify the content validity of the instrument from expert-judge
ratings using **Aiken's V coefficient** and its 95% confidence interval
(Penfield & Giacobbi, 2004), reported separately for the two content dimensions
evaluated by the judges: **Relevance** and **Wording**.

**Data.**
- `demo_judges.csv` — sociodemographic profile of the expert judges.
- `data_Aiken_NPS.csv` — expert ratings per item and content dimension (4-point scale).

**Required packages:** `pandas`, `numpy`, `scipy`, `matplotlib`, `seaborn`, `tabulate`.


## 1. Setup and data import

In [ ]:
# Google Colab only: mount Google Drive to access the dataset files.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
from scipy.stats import norm

# Plotting and table/markdown rendering
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
from IPython.display import display, Markdown

# Load the expert-judge datasets from Google Drive
#   df  -> sociodemographic profile of the expert judges
#   df2 -> expert ratings per item and content dimension
df  = pd.read_csv('/content/drive/MyDrive/NPS/demo_judges.csv')
df2 = pd.read_csv('/content/drive/MyDrive/NPS/data_Aiken_NPS.csv')


## 2. Characterization of the expert judges

In [ ]:
# Total number of expert judges
n = len(df)

# Container for the summary-table rows
summary_table = []

# Helper: append a categorical variable as a block of (category, frequency) rows
def add_block(variable, variable_name):
    summary_table.append([f"**{variable_name}**", "", ""])
    counts = df[variable].value_counts().sort_index()
    for category, freq in counts.items():
        percentage = round((freq / n) * 100, 1)
        summary_table.append(["", f"{category}", f"{freq} ({percentage}%)"])

# Categorical variables
add_block("sexo",                    "Sex")
add_block("grado_academico",         "Highest academic degree")
add_block("especializacion",         "Specialization")
add_block("experiencia_profesional", "Type of (main) professional experience")

# Numeric variable: years of experience (median/IQR, mean/SD, range)
summary_table.append(["**Years of experience**", "", ""])
summary_table.append(["", "Median (Q1, Q3)",
                      f"{df['anios_experiencia'].median()} "
                      f"({df['anios_experiencia'].quantile(0.25)}, "
                      f"{df['anios_experiencia'].quantile(0.75)})"])
summary_table.append(["", "Mean (SD)",
                      f"{round(df['anios_experiencia'].mean(),1)} "
                      f"({round(df['anios_experiencia'].std(),1)})"])
summary_table.append(["", "Min, Max",
                      f"{df['anios_experiencia'].min()}, {df['anios_experiencia'].max()}"])

# Experience in instrument validation
add_block("exp_valid_inst", "Do you have experience in instrument validation?")

# Assemble and display the table
table_df = pd.DataFrame(summary_table, columns=["Variable", "Category", f"n={n}"])

print("### Table 3. Characterization of expert judges\n")
print(tabulate(table_df, headers="keys", tablefmt="github", showindex=False))


## 3. Statistical functions: Aiken's V and confidence interval

Aiken's V summarizes expert agreement on each item. Its confidence interval is
obtained with the score-based formula of Penfield & Giacobbi (2004).

In [ ]:
def compute_aiken_v(values, k):
    """
    Compute Aiken's V coefficient for content validity.

    Parameters
    ----------
    values : array-like
        Expert ratings for a given item (integers from 1 to k).
    k : int
        Maximum value of the rating scale (e.g., 4 for a 4-point Likert scale).

    Returns
    -------
    float
        Aiken's V coefficient.
    """
    N = len(values)                       # number of expert raters
    V = sum(values - 1) / (N * (k - 1))   # Aiken's formula
    return V


def compute_aiken_ci(V, n, k, confidence=0.95):
    """
    Confidence interval for Aiken's V (Penfield & Giacobbi, 2004).

    Parameters
    ----------
    V : float
        Aiken's V value.
    n : int
        Number of raters.
    k : int
        Number of points on the rating scale.
    confidence : float, optional
        Desired confidence level (default = 0.95).

    Returns
    -------
    tuple of float
        Lower (L) and upper (U) bounds of the confidence interval.
    """
    z = norm.ppf(1 - (1 - confidence) / 2)            # z-score for the CI level

    # Term inside the square root of the CI formula
    sqrt_term = np.sqrt(4 * n * k * V * (1 - V) + z**2)

    # Lower (L) and upper (U) bounds
    L = (2 * n * k * V + z**2 - z * sqrt_term) / (2 * (n * k + z**2))
    U = (2 * n * k * V + z**2 + z * sqrt_term) / (2 * (n * k + z**2))

    return (L, U)


## 4. Aiken's V by item

In [ ]:
# Reshape to long format for item-level analysis
lf2 = df2.melt(id_vars='dimension', var_name='item', value_name='valor')

# Rating-scale points (4-point scale) and confidence level
k         = 4
confianza = 0.95

# Mean rating and number of raters per item-dimension combination
resultados = (
    lf2.groupby(['dimension', 'item'])
    .agg(promedio=('valor', 'mean'), n=('valor', 'count'))
    .reset_index()
)

# Aiken's V per item
resultados['V'] = (
    lf2.groupby(['dimension', 'item'])['valor']
    .apply(lambda x: compute_aiken_v(x.values, k))
    .values
)

# Confidence interval for Aiken's V per item
resultados[['lim_inf', 'lim_sup']] = resultados.apply(
    lambda row: pd.Series(compute_aiken_ci(row['V'], row['n'], k, confianza)),
    axis=1
)

# Frequency of each response value (1..k) per item
frecuencias = (
    lf2.groupby(['dimension', 'item', 'valor'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Merge frequencies with the Aiken's V results
resultados_final = pd.merge(
    resultados, frecuencias, on=['dimension', 'item'], how='left'
).fillna(0)

# Ensure a frequency column for every scale point (1..k); fill absent ones with 0
# so the table always shows all response options even if one is never selected.
for value in range(1, k + 1):
    if value not in resultados_final.columns:
        resultados_final[value] = 0

# Column order for presentation: dimension, item, frequencies, mean, V, CI
cols_order = (['dimension', 'item'] + list(range(1, k + 1))
              + ['promedio', 'V', 'lim_inf', 'lim_sup'])
resultados_final = resultados_final[cols_order]

# Round numeric columns to three decimals
resultados_final[['promedio', 'V', 'lim_inf', 'lim_sup']] = (
    resultados_final[['promedio', 'V', 'lim_inf', 'lim_sup']].round(3)
)

# Display the final results table
display(Markdown("### Final results of Aiken's V analysis"))
display(resultados_final)


## 5. Visualization of Aiken's V and confidence intervals

In [ ]:
# Display labels for the two content dimensions
titulo_dim = {
    'relevance': 'Relevance',
    'wording':   'Wording'
}

# Black-and-white style suitable for publication
plt.rcParams.update({
    'axes.edgecolor': 'black',
    'xtick.color':    'black',
    'ytick.color':    'black',
    'text.color':     'black',
    'axes.labelcolor':'black'
})

# Set the figure height dynamically based on the number of items
n_items         = resultados['item'].nunique()
height_per_item = 0.3                       # adjust for more/less vertical spacing
fig_height      = max(7.2, n_items * height_per_item)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(9.6, fig_height), sharex=True)

for ax, dimension in zip(axes, ['relevance', 'wording']):
    subset = resultados[resultados['dimension'] == dimension].copy()
    subset = subset.reset_index(drop=True)
    subset['y_pos'] = subset.index

    # Point estimate with the 95% confidence interval as error bars
    ax.errorbar(
        subset['V'], subset['y_pos'],
        xerr=[subset['V'] - subset['lim_inf'], subset['lim_sup'] - subset['V']],
        fmt='o', color='black', ecolor='gray', elinewidth=1, capsize=3, markersize=5
    )

    # Decision thresholds
    ax.axvline(0.8, color='black', linestyle='--', linewidth=1,
               label='V >= 0.8 (acceptance criterion)')
    ax.axvline(0.5, color='black', linestyle=':', linewidth=1,
               label='Lower limit >= 0.5 (minimum criterion)')

    # Annotate each point with its V value
    for _, row in subset.iterrows():
        ax.text(
            row['V'], row['y_pos'] - 0.35, f"{row['V']:.2f}",
            ha='center', fontsize=8, color='black', weight='bold',
            bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.15')
        )

    ax.set_yticks(subset['y_pos'])
    ax.set_yticklabels(subset['item'], fontsize=9)
    ax.set_xlabel(f"Aiken's V - {titulo_dim[dimension]}", fontsize=12)
    ax.set_ylabel("Item", fontsize=10)
    ax.invert_yaxis()

    x_max = max(subset['lim_sup'].max(), 1.0) + 0.03
    ax.set_xlim(0.3, x_max)

    ax.legend(loc='upper left', fontsize=8, frameon=False)

plt.tight_layout()

# Export the figure for the manuscript (vector + high-resolution raster)
plt.savefig("V_Aiken_final.pdf", format='pdf', bbox_inches='tight')
plt.savefig("V_Aiken_final.png", format='png', dpi=600, bbox_inches='tight')
plt.show()


## 6. Reproducibility notes

- Designed to run in [Google Colab](https://colab.research.google.com/) (Python 3).
- Required packages: `pandas`, `numpy`, `scipy`, `matplotlib`, `seaborn`, `tabulate`.
- The repository should include this `.ipynb` notebook together with the
  `demo_judges.csv` and `data_Aiken_NPS.csv` data files for full reproducibility.

---

© 2026 R. Monge and co-authors. Released under the
[MIT License](https://opensource.org/licenses/MIT).
